# Step 1: vLLM 加载量化模型 — auto 识别 + quantization_config 构造 + 声明式边界

**目标**：建立 M4 声明式部署的核心认知——不止让 vLLM "读懂" 量化 config，而是理解 `quantization_config` 结构后**能自己写出来**：谁适配谁、vLLM 加载量化模型内部经历哪几步、config 字段如何驱动 vLLM 选 kernel、声明式部署的两个前提与三条件。最后能写出 FP8/AWQ/SmoothQuant 三种 config。

**对应 OUTLINE 课时**：4.1 加载识别 + flag/kernel 速查 + 声明式边界（~50 分钟）。

> **单 env**：本 notebook 在 `course/m4-deploy-loop` 单 vllm env 跑。L3 的 7B 量化产物跨模块引用 M2 `out/`（先跑完 M2 s1/s3/s5 产出 FP8/AWQ/SmoothQuant 7B）；**0.5B 路径不需此前置**（L2 用内联 config 样本兜底）。


## 学完应能讲清（学完本节应能口头回答）

1. **谁适配谁**？"适配" 是 vLLM 读你的 `quantization_config` 适配你的模型，**不是你适配 vLLM**。你产出量化产物后，从 `vllm serve` 到服务起来，vLLM 替你做了什么？（提示：选架构实现类 + 选量化 kernel，你不写任何适配代码）
2. vLLM 加载量化模型内部经历哪几步？`quantization_config` 的哪个字段驱动 vLLM 给某层选哪个 kernel？（`targets`→哪些层、`scheme`→哪个 kernel、`group_size`→权重分组粒度、`ignore`→跳过哪些层）
3. compressed-tensors 的 `quantization_config` 有哪些关键字段（`config_groups`/`targets`/`scheme`/`num_bits`/`group_size`/`ignore`）？给定一个量化需求（如 "FP8 量化除 lm_head 外所有 Linear 层"），你能写出对应结构吗？
4. 声明式部署的**两个前提**（架构 vLLM 已支持 + 量化用标准 scheme）+ **三条件**（标准 scheme + 权重布局符合 kernel 契约 + vLLM 补了该架构 quantized 层）；`No compatible kernel found` 对应哪个不满足？
5. 三方法（FP8/AWQ/SmoothQuant）各走哪个 kernel？为什么多数情况**不传 `--quantization` flag** 反而对？遗留 AutoAWQ 为什么 flag 是 `auto_awq`（带下划线）不是 `awq`？


In [ ]:
%%capture
import pathlib, os, json
import ipytest
ipytest.autoconfig()


In [ ]:
# Setup cell（cwd 无关路径解析）。M4 跨模块读 M2/M3 的 7B 量化产物做 L3；0.5B 兜底 L2 流程。
import pathlib

def _find_module_root(start):
    p = pathlib.Path(start).resolve()
    for cand in [p, *p.parents]:
        if (cand / "scripts").is_dir() and (cand / "steps").is_dir():
            return cand
    raise RuntimeError("找不到模块根（含 scripts/ + steps/ 的目录）")

MODULE_ROOT   = _find_module_root(pathlib.Path.cwd())
MODEL_DIR      = MODULE_ROOT / "models" / "Qwen2.5-7B-Instruct"          # FP16 基线（s3 压测对比用）
TINY_MODEL_DIR = MODULE_ROOT / "models" / "Qwen2.5-0.5B-Instruct"         # L2/L3 轻量验证兜底
OUT_ROOT       = MODULE_ROOT / "out"; OUT_ROOT.mkdir(parents=True, exist_ok=True)
# 跨模块引用 M2/M3 的 7B 量化产物（兄弟模块 out/，只读不写）
REPO_COURSE = MODULE_ROOT.parent
M2_OUT = REPO_COURSE / "m2-quant-pipeline" / "out"      # qwen7b-fp8 / qwen7b-awq / qwen7b-smoothquant
M3_OUT = REPO_COURSE / "m3-tuning-eval" / "out"         # 调优后的 mixed-precision 产物
print("MODULE_ROOT =", MODULE_ROOT)
print("M2_OUT =", M2_OUT, "| exists:", M2_OUT.exists())
print("M3_OUT =", M3_OUT, "| exists:", M3_OUT.exists())


## 原理：声明式部署 = "你产出产物，vLLM 读 config 自动适配"

声明式部署的核心是**谁适配谁**——是 **vLLM 读你的 `quantization_config` 适配你的模型**，不是你去适配 vLLM。你只负责"产出正确的量化产物"（标准 scheme 量化 → 正确 packed 权重 + 把 `quantization_config` 写进 `config.json`），vLLM 读它自动选架构实现 + 量化 kernel——你夹在中间**不写任何适配代码**。llm-compressor 的 `save_pretrained` 自动把 `quantization_config` 写进 `config.json`，这就是"声明"的来源。

**vLLM 适配全过程（内部 6 步，`vllm serve` 启动时自动做）**：

1. 读 `config.json` 的 `architectures`（如 `Qwen2ForCausalLM`）→ 查 `ModelRegistry` 找架构实现类（attention/MLP/forward）。**层①架构适配**。
2. 读 `quantization_config`（`quant_method`）→ 查 quantization registry 找量化方法类 + 解析 `config_groups` 确定"哪些层用哪种 scheme"。**层②量化适配**。
3. 遍历模型层，给被量化的层套对应 kernel wrapper（`targets` 决定哪些层、`scheme` 决定 FP8/AWQ/INT8 哪个 kernel）。
4. 加载 packed 权重 + scale/zero_point 张量塞进 wrapper（`group_size` 决定分组粒度）。
5. **profile run**（跑 dummy 输入探测 KV-Cache 块数）——**这就是"验证"**：vLLM 启动自带、不是你写的；报错（`No compatible kernel found` / OOM）= 三条件某个不满足。
6. 起服务接请求。

**config 字段 → vLLM 适配步骤的映射**（本节构造填空的认知基础——懂了这个映射，才能从"量化需求"推导出"该写什么 config"，而非机械抄 JSON）：

| `quantization_config` 字段 | 驱动 vLLM 哪步 |
|---|---|
| `targets` | 步骤③ 哪些层套 kernel wrapper |
| `scheme`（FP8 / W4A16 / W8A8）| 步骤③ 选哪个 kernel |
| `num_bits` / `group_size` | 步骤③-④ 权重打包粒度 |
| `ignore` | 步骤③ 跳过哪些层（保持高精度）|

### Kernel 速查表：三方法各走哪个 vLLM kernel

| 量化方法 | vLLM kernel | 说明 |
|---|---|---|
| FP8 (W8A8) | CUTLASS `torch._scaled_mm` | Hopper 原生 FP8 矩阵乘（SM90+）；`--quantization fp8` 可显式指定，但默认 `auto` 即走此 |
| AWQ (W4A16) | Marlin / Machete | W4A16 mixed-input kernel（权重 INT4、激活 FP16）；Marlin 通用，Machete 是更新的 W4A16 专用 kernel |
| SmoothQuant (W8A8 INT8) | INT8-Marlin / scaled_mm | W8A8 INT8 kernel；Hopper 上 `scaled_mm`、非 Hopper 回退 INT8-Marlin |
| 遗留 AutoAWQ | awq_marlin | 非 compressed-tensors 格式，需显式传 `--quantization auto_awq`（带下划线，见下方说明） |

> **为什么这很重要**：kernel 名原只在 `pick_vllm_flag_and_kernel` 的 docstring 里。把这张表写进原理讲解，让你读完即能回答 Q5（"三方法各走哪个 kernel"）——不需要跳到填空 docstring 才看到答案。

> **为什么 `auto_awq` 带下划线？** vLLM 用 `auto_awq`（带下划线）区分**遗留 AutoAWQ 库产物** vs **标准 compressed-tensors AWQ**——二者是 vLLM 两个独立的 quantization registry 项；标准 compressed-tensors AWQ（llm-compressor 产出）走 `auto` 无需 flag，只有遗留 AutoAWQ 格式才需 `--quantization auto_awq`。学员常把 `awq` 和 `auto_awq` 混淆——认知要点：**不是所有 AWQ 都要传 flag**，标准 compressed-tensors AWQ 不传 flag（auto 自动识别）。

### 声明式的前提：两层适配 + 三条件

声明式承诺 = "**不用写模型适配代码、不用写反量化代码**"（前提架构已通）——成立。但**不是**"随便量化都能跑"——还得"把量化做对"。

**两层适配**（vLLM 加载量化模型要两层都通）：
```
config.json
 ├─ architectures: XxxForCausalLM   →  层①架构适配（vLLM 有没有这个模型的实现类）
 └─ quantization_config: {...}       →  层②量化适配（这个 scheme vLLM 认不认）
```

**三条件**（bf16 能跑的模型，量化后直接声明式跑通需满足）：
1. **量化方案是 vLLM 认的标准 scheme**（compressed-tensors 的 FP8/AWQ/INT8、GPTQ 等，用 llm-compressor 产出）。
2. **权重布局符合 kernel 契约**（粒度/scale 形状/group_size 整除/AWQ `zero_point=False` 等）。
3. **vLLM 给该架构补了该量化的 kernel 路径**（新架构常只实现 bf16 forward、未补 quantized 层）。

**关键结论**：
- **量化不创造架构适配**：vLLM 未适配的架构，量化前后都跑不了原生路径。
- `No compatible kernel found` 通常对应条件③不满足（架构没补 quantized kernel）或条件②（权重布局不符）。

**边界外三条**（声明式不成立，仅讲不实操）：(a) 多模态变体（可能需 trust-remote-code）；(b) 非标准/自定义 quantization scheme；(c) vLLM 未原生支持的冷门架构（走 Transformers fallback `--model-impl transformers`，性能损失大，或写 model 适配 = 改代码）。


## 亲手摸一摸：真实 7B 量化产物的 quantization_config

看 M2 产的 FP8/AWQ/SmoothQuant 三种 `config_groups` 字段差异——理解"哪些层用什么 scheme"这个结构是 vLLM 步骤②③适配的依据，而不是把 config 当黑盒。


In [ ]:
# 摸一摸：打印三方法 7B 量化产物的 quantization_config（M2 out/，跨模块只读）
# 缺产物时用内联样本兜底（L2 独立可跑）。注意：SAMPLE_CONFIGS 是「完整 config dict」
# （含 quantization_config 键），与真实 config.json 同构——detect_quant_scheme 吃完整 config。
SAMPLE_CONFIGS = {
    "FP8": {"architectures": ["Qwen2ForCausalLM"], "quantization_config": {
        "config_groups": {"group_0": {"format": "float-quantized",
            "targets": ["Linear"],
            "weights": {"num_bits": 8, "type": "float", "strategy": "channel", "symmetric": True},
            "input_activations": {"num_bits": 8, "type": "float", "strategy": "token", "symmetric": True, "dynamic": True}}},
        "ignore": ["lm_head"], "quant_method": "compressed-tensors"}},
    "W4A16 (AWQ)": {"architectures": ["Qwen2ForCausalLM"], "quantization_config": {
        "config_groups": {"group_0": {"format": "pack-quantized",
            "targets": ["Linear"],
            "weights": {"num_bits": 4, "type": "int", "strategy": "group", "group_size": 128, "symmetric": False},
            "input_activations": None}},
        "ignore": ["lm_head"], "quant_method": "compressed-tensors"}},
    "W8A8 (SmoothQuant)": {"architectures": ["Qwen2ForCausalLM"], "quantization_config": {
        "config_groups": {"group_0": {"format": "int-quantized",
            "targets": ["Linear"],
            "weights": {"num_bits": 8, "type": "int", "strategy": "channel", "symmetric": True},
            "input_activations": {"num_bits": 8, "type": "int", "strategy": "token", "symmetric": True, "dynamic": True}}},
        "ignore": ["lm_head"], "quant_method": "compressed-tensors"}},
}

real = {"fp8": M2_OUT / "qwen7b-fp8", "awq": M2_OUT / "qwen7b-awq", "smoothquant": M2_OUT / "qwen7b-smoothquant"}
print("=== 三方法 config_groups 字段差异（vLLM 步骤②③的输入）===")
for name, cfg in SAMPLE_CONFIGS.items():
    qc = cfg["quantization_config"]
    g0 = qc["config_groups"]["group_0"]
    w = g0["weights"]; act = g0.get("input_activations")
    print("\n[%s] targets=%s ignore=%s" % (name, g0['targets'], qc['ignore']))
    print("  weights: type=%s num_bits=%s strategy=%s group_size=%s symmetric=%s" % (
        w['type'], w['num_bits'], w['strategy'], w.get('group_size'), w['symmetric']))
    if act is None:
        print("  input_activations: None (weight-only，激活全程 FP16)")
    else:
        print("  input_activations: type=%s dynamic=%s" % (act['type'], act.get('dynamic')))
print("\n（真 7B 产物路径见 L3）:", real)


## 本步填空（3 个，理解-构造-速查递进）

1. **`detect_quant_scheme(model_path_or_config)`** — 读 `config.json` 的 `quantization_config`，返回结构化信息 dict（scheme + targets + 粒度），FP16 返回 None。**为什么这么设计（填前先想）**：先"读懂"——亲手解析 `config_groups`，理解"哪些层用什么 scheme"这个结构是 vLLM 步骤②③适配的依据，而不是把 config 当黑盒。
2. **`build_quantization_config(scheme, targets, num_bits=None, group_size=None, ignore=())`**（**理解型核心**）— **构造** compressed-tensors 的 `quantization_config` 字典（`config_groups` + `ignore` 结构）。**为什么这么设计**：这是"自己写出 config"——从量化需求（如 "FP8 量化除 lm_head 外所有 Linear 层" / "AWQ W4A16 group_size=128"）推导出正确字段结构。**docstring 只给字段语义和约束**（`group_size` 须整除 hidden_size、`ignore` 决定哪些层保持高精度、`targets` 用 `['Linear']` 匹配），**不给逐字实参**——你要理解每个字段驱动 vLLM 哪步、该填什么值。
3. **`pick_vllm_flag_and_kernel(scheme)`**（判断型）— 给 scheme 返回 `(vllm_flag_or_None, kernel_name)` 速查（FP8→(None/CUTLASS scaled_mm)、AWQ→(None/Marlin/Machete)、SmoothQuant INT8→(None/INT8-Marlin)、遗留 AutoAWQ→('auto_awq'/awq_marlin)）。**为什么这么设计**：scheme→flag/kernel 速查逻辑化；多数"不传 flag"体现 auto 的便利，`auto_awq` 下划线是高频坑。


In [ ]:
def detect_quant_scheme(model_path_or_config):
    """读 config.json 的 quantization_config，返回结构化信息 dict 或 None（FP16/未压缩）。
    输入：path（str/Path 指向 config.json 或模型目录），或已加载的 config dict。

    为什么这么设计（填前先想）：
    - vLLM 步骤②就是读 quantization_config 解析 config_groups——你亲手解析一遍，
      才理解 scheme 不是单一字段，而是由 weights/input_activations 的 type/num_bits/strategy 组合推断。
    - scheme 推断规则（从真实 M2 产物归纳）：
        * weights.type='float' + act.type='float'        -> 'FP8'
        * weights.type='int'  + act 存在(int)            -> 'W8A8'  (SmoothQuant INT8)
        * weights.type='int'  + act 为 None              -> 'W4A16' (AWQ weight-only)
    - FP16 / 无 quantization_config / quant_method != 'compressed-tensors' -> 返回 None。

    返回（None 或 dict，dict 含：scheme, targets, num_bits, group_size, ignore, quant_method）。
    """
    # TODO:
    #   1) 输入归一：dict 直接用；否则把 path 解析到 config.json 并 json.load。
    #   2) 取 quantization_config；为空 或 quant_method != 'compressed-tensors' -> return None。
    #   3) 取 config_groups 的 group_0（或第一个 group）。
    #   4) 按上面规则推断 scheme；从 weights 取 num_bits/group_size；从 qc 取 ignore（默认 []）。
    #   5) 返回 dict(scheme=, targets=, num_bits=, group_size=, ignore=, quant_method=)。
    raise NotImplementedError


In [ ]:
def build_quantization_config(scheme, targets, num_bits=None, group_size=None, ignore=()):
    """构造 compressed-tensors 的 quantization_config 字典。

    参数（语义，不给逐字实参）：
    - scheme: 'FP8'（FP8_DYNAMIC）/ 'W4A16'（AWQ weight-only）/ 'W8A8'（SmoothQuant INT8）
    - targets: list/tuple，如 ['Linear']（驱动 vLLM 步骤③ 哪些层套 wrapper）
    - num_bits: W4A16 默认 4、W8A8/FP8 默认 8（驱动权重打包位数）
    - group_size: 仅 per-group（W4A16）有意义；W4A16 必须给（否则 ValueError）；
                  W8A8 是 per-channel，不应有 group_size（给了 >0 的值要 ValueError）
    - ignore: tuple/list，如 ('lm_head',)（驱动 vLLM 步骤③ 跳过哪些层保持高精度）

    为什么这么设计（填前先想）：
    - 每个字段都映射到 vLLM 的一步——你不是在抄 JSON，是在"指挥 vLLM 适配"。
    - 三 scheme 的权重/激活结构差异（从摸一摸 cell 真实产物归纳）：
        FP8  : weights=float/channel/sym + act=float/token/sym/dynamic；format='float-quantized'
        W4A16: weights=int/group/sym=False/group_size/zp_dtype='torch.int8' + act=None；format='pack-quantized'
        W8A8 : weights=int/channel/sym + act=int/token/sym/dynamic；format='int-quantized'
    - 输出顶层：{'config_groups': {'group_0': {...}}, 'quant_method': 'compressed-tensors',
                'format': <fmt>, 'ignore': list(ignore)}
    """
    # TODO:
    #   1) targets / ignore 先转 list。
    #   2) 按 scheme 分支构造 weights dict（含 type/strategy/symmetric/dynamic/num_bits/group_size/zp_dtype）：
    #      FP8/W8A8 还要构造 input_activations dict（token 策略、dynamic=True）；W4A16 的 act=None。
    #      W4A16 缺 group_size 报错；W8A8 给了正 group_size 报错。
    #      num_bits 缺省：FP8/W8A8->8，W4A16->4。
    #   3) group_0 = {'targets': targets, 'weights': w, 'input_activations': act,
    #                 'output_activations': None, 'format': fmt}
    #   4) 返回顶层 dict（config_groups + quant_method + format + ignore）。
    raise NotImplementedError


In [ ]:
def pick_vllm_flag_and_kernel(scheme):
    """给 scheme 返回 (vllm_flag_or_None, kernel_name)。

    为什么这么设计（填前先想）：
    - vLLM --quantization 默认 'auto'：读 config.json 的 quantization_config 自动识别 compressed-tensors
      格式——所以标准 FP8/AWQ/SmoothQuant 产物 flag 都是 None（不传，让 auto 工作）。
    - 遗留 AutoAWQ 产物（非 compressed-tensors）必须显式传 --quantization auto_awq
      （注意下划线！不是 'awq'，这是高频坑——vLLM registry 里的方法名带下划线）。
    - kernel 名是 vLLM 给该 scheme 选的具体实现（教学认知，知道有哪些 kernel 即可）。

    返回映射（scheme 不在表内可 raise KeyError）：
      'FP8'      -> (None,            'CUTLASS scaled_mm (FP8)')
      'W4A16'    -> (None,            'AWQ-Marlin / Machete (W4A16)')
      'W8A8'     -> (None,            'INT8-Marlin (W8A8)')
      'auto_awq' -> ('auto_awq',      'awq_marlin (遗留 AutoAWQ)')
    """
    # TODO: 用 dict 映射 4 个 scheme -> (flag, kernel) 元组；未知 scheme raise。
    raise NotImplementedError


In [ ]:
%%ipytest -qq

def test_detect_fp8():
    r = detect_quant_scheme(SAMPLE_CONFIGS["FP8"])
    assert r is not None and r["scheme"] == "FP8"
    assert r["targets"] == ["Linear"]
    assert r["num_bits"] == 8
    assert r["ignore"] == ["lm_head"]

def test_detect_awq():
    r = detect_quant_scheme(SAMPLE_CONFIGS["W4A16 (AWQ)"])
    assert r["scheme"] == "W4A16"
    assert r["group_size"] == 128 and r["num_bits"] == 4

def test_detect_smoothquant():
    r = detect_quant_scheme(SAMPLE_CONFIGS["W8A8 (SmoothQuant)"])
    assert r["scheme"] == "W8A8" and r["num_bits"] == 8

def test_detect_fp16_returns_none():
    # FP16 基线：无 quantization_config
    assert detect_quant_scheme({"architectures": ["Qwen2ForCausalLM"]}) is None
    # quant_method 非 compressed-tensors 也 None
    assert detect_quant_scheme({"quantization_config": {"quant_method": "bnb"}}) is None

def test_build_fp8_all_linear():
    qc = build_quantization_config("FP8", ["Linear"], ignore=("lm_head",))
    assert qc["quant_method"] == "compressed-tensors"
    assert qc["ignore"] == ["lm_head"]
    g0 = qc["config_groups"]["group_0"]
    assert g0["targets"] == ["Linear"]
    assert g0["weights"]["type"] == "float" and g0["weights"]["num_bits"] == 8
    assert g0["input_activations"]["dynamic"] is True

def test_build_awq_group_size():
    qc = build_quantization_config("W4A16", ["Linear"], group_size=128, ignore=["lm_head"])
    g0 = qc["config_groups"]["group_0"]
    assert g0["weights"]["num_bits"] == 4 and g0["weights"]["group_size"] == 128
    assert g0["weights"]["symmetric"] is False
    assert g0["input_activations"] is None  # weight-only
    assert g0["format"] == "pack-quantized"

def test_build_awq_missing_group_size_raises():
    import pytest
    with pytest.raises(ValueError):
        build_quantization_config("W4A16", ["Linear"])

def test_build_w8a8_rejects_group_size():
    import pytest
    with pytest.raises(ValueError):
        build_quantization_config("W8A8", ["Linear"], group_size=128)

def test_pick_flag_and_kernel():
    assert pick_vllm_flag_and_kernel("FP8") == (None, "CUTLASS scaled_mm (FP8)")
    assert pick_vllm_flag_and_kernel("W4A16")[0] is None   # auto 识别，不传 flag
    assert pick_vllm_flag_and_kernel("auto_awq") == ("auto_awq", "awq_marlin (遗留 AutoAWQ)")
    # 关键坑：遗留 AutoAWQ flag 带 _ 下划线
    assert pick_vllm_flag_and_kernel("auto_awq")[0] == "auto_awq"


## L2（CPU）：解析真实 config 样本验逻辑

L2 验 `detect_quant_scheme` 对真实产物/内联样本的解析正确性（CPU 可跑，不需 vllm 真加载）。优先解析跨模块 M2 7B 量化产物；缺产物用内联 `SAMPLE_CONFIGS` 兜底（保证流程层独立可跑）。


In [ ]:
## L2：解析真 7B 产物（M2 out/）验 detect；缺产物用内联样本兜底
import pathlib
real_dirs = {"FP8": M2_OUT / "qwen7b-fp8", "AWQ": M2_OUT / "qwen7b-awq", "SmoothQuant": M2_OUT / "qwen7b-smoothquant"}
fallback = {"FP8": SAMPLE_CONFIGS["FP8"], "AWQ": SAMPLE_CONFIGS["W4A16 (AWQ)"], "SmoothQuant": SAMPLE_CONFIGS["W8A8 (SmoothQuant)"]}

print("=== L2：detect_quant_scheme 解析三方法 ===")
for name in ["FP8", "AWQ", "SmoothQuant"]:
    d = real_dirs[name]
    if (d / "config.json").exists():
        cfg = json.loads((d / "config.json").read_text())
        src = "M2 out/" + d.name + "（真 7B 产物）"
    else:
        cfg = fallback[name]
        src = "内联 SAMPLE（M2 产物缺失，用兜底样本）"
    r = detect_quant_scheme(cfg)
    print("\n[" + name + "] 来源=" + src)
    print("  -> scheme={} targets={} num_bits={} group_size={} ignore={}".format(
        r["scheme"], r["targets"], r["num_bits"], r["group_size"], r["ignore"]))
# 验：0.5B FP16 基线必须返回 None
fp16 = json.loads((TINY_MODEL_DIR / "config.json").read_text())
assert detect_quant_scheme(fp16) is None, "FP16 基线应返回 None"
print("\nL2 通过：FP16->None、三方法 config_groups 结构解析正确（vLLM 步骤②③依据）。")
print("（真 vllm LLM().generate() 验证 auto 识别 + kernel 选择见 L3）")


## L3（H200，GPU + SKIP_L3 双守卫）：0.5B 离线 generate + 7B 三方法 LLM 加载

L3 验 vLLM 真能 auto 识别 compressed-tensors 产物并选对 kernel：先在 0.5B 上离线 `vllm.LLM().generate()` 跑一句（轻量），再对 7B 三方法产物各 `LLM()` 加载 generate。

> **L3 双守卫**：`torch.cuda.is_available() and not os.environ.get('SKIP_L3')`——reviewer 执行验证设 `SKIP_L3=1` 跳过（7B serve 分钟级，太重）；真人/学员跑时不设，L3 实证。


In [ ]:
import torch, os
def run_l3():
    # 先 0.5B 离线 LLM（轻量，验证 vllm 流程通）
    from vllm import LLM, SamplingParams
    print("[L3-0.5B] 离线 LLM().generate() 验证 vLLM 加载流程（FP16 基线）...")
    llm = LLM(model=str(TINY_MODEL_DIR), dtype="float16", enforce_eager=True, gpu_memory_utilization=0.5)
    out = llm.generate(["你好，用一句话介绍量化。"], SamplingParams(max_tokens=16, temperature=0))
    print("  0.5B 输出:", out[0].outputs[0].text.strip())
    del llm

    # 7B 三方法：各 LLM() 加载验 auto 识别 + kernel 选择（跨模块 M2 out/ 产物）
    candidates = {"FP8": M2_OUT / "qwen7b-fp8", "AWQ": M2_OUT / "qwen7b-awq", "SmoothQuant": M2_OUT / "qwen7b-smoothquant"}
    for name, d in candidates.items():
        if not (d / "config.json").exists():
            print("\n[{}] {} 不存在——先跑 M2 对应步骤产出 7B 量化模型。".format(name, d))
            continue
        print("\n[L3-7B-{}] LLM() 加载 {}，验 auto 识别 compressed-tensors...".format(name, d))
        llm = LLM(model=str(d), enforce_eager=True, gpu_memory_utilization=0.9)
        o = llm.generate(["2+2=?"], SamplingParams(max_tokens=8, temperature=0))
        print("  {} 输出: {} | 加载成功=vLLM auto 选对 kernel".format(name, o[0].outputs[0].text.strip()))
        del llm

if torch.cuda.is_available() and not os.environ.get('SKIP_L3'):
    run_l3()
else:
    print("跳过 L3：无 GPU 或 SKIP_L3=1（reviewer 执行验证跳过真 7B vllm 跑；真人跑时不设 SKIP_L3，L3 实证）。")


## 产物检查：auto 识别成功 = 声明式闭环成立

L3 跑通即证明：vLLM 读 `config.json` 的 `quantization_config` 自动识别 compressed-tensors 格式 → 选对应 kernel → 正常 generate——**全程无需改模型代码、无需传 `--quantization` flag**（标准 scheme 走 auto）。这就是声明式部署的闭环。

**回顾三条件**（L3 跑通 = 三条件全满足）：
1. scheme 是标准 compressed-tensors（FP8/AWQ/SmoothQuant）✅
2. 权重布局符合 kernel 契约（group_size 整除、AWQ zero_point 符合 Marlin）✅
3. vLLM 给 Qwen2ForCausalLM 补了 quantized kernel 路径 ✅

若 L3 报 `No compatible kernel found`——回 §5 三条件排查（多半是条件③：架构没补 quantized kernel，或条件②：权重布局不符）。s4 会专门讲报错诊断。
